# Module 3: Deploy — LangGraph CLI + LangSmith Deployments

> Part of the **Modular Workshops** series. Standalone, ~15 min.

Ship the deep agent in `agents/deep_agent/` to **LangSmith Deployments** with the `langgraph` CLI. The deployed agent gets 30+ endpoints out of the box — REST, MCP, A2A, Agent Protocol, HITL, memory, and Studio UI.

**What you'll do:**
1. Inspect the deployable layout (`langgraph.json` + `agents/deep_agent/`)
2. Run it locally with `langgraph dev`
3. Validate the config, then `langgraph deploy` to LangSmith


## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os
print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:   ", bool(os.environ.get("TAVILY_API_KEY")))


LANGSMITH_API_KEY set: True
OPENAI_API_KEY set:    True
TAVILY_API_KEY set:    True


## 1. Project Structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`.


In [2]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


langgraph.json (workshop root)
---
deep_agent/
  deepagents.toml
  agent.py
  AGENTS.md
  skills/
    linkedin-post/
      SKILL.md
    twitter-post/
      SKILL.md


## 2. Configuration

`langgraph.json` is the deploy config. Minimal fields: `dependencies`, `graphs`, and `env`. The LangGraph CLI uses it for both `langgraph dev` and `langgraph deploy`.

`AGENTS.md` is the agent's identity — loaded into the prompt via the `memory=` argument in `create_deep_agent()` (see `agent.py`) and editable by the agent itself at runtime.


In [3]:
# langgraph.json -- the deploy configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


langgraph.json:
{
  "dependencies": ["."],
  "graphs": {
    "deep_agent": "./agents/deep_agent/agent.py:agent"
  },
  "env": ".env"
}

AGENTS.md:
# Research Assistant

You are an expert research assistant that can search the web, synthesize findings, and produce polished reports and content.

## Workflow

1. **Plan** — Use `write_todos` to break the task into steps
2. **Research** — Delegate research to the `research-agent` using the `task()` tool
3. **Synthesize** — Combine findings into a comprehensive report
4. **Write** — Save the final report to `/final_report.md`
5. **Remember** — Save key takeaways to `/memories/research_notes.md` for future reference

## Rules

- Delegate research to the research-agent rather than searching directly
- After receiving research results, synthesize and write the report yourself
- Consolidate citations — each unique URL gets one number [1], [2], [3]
- End reports with a Sources section listing all referenced URLs
- Check for relevant skills when a

The agent itself lives in `agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 3. Local Development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith (beta)
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 4. Validate the Config

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch config and import errors before deploying.


In [4]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


Configuration file /Users/robertxu/Desktop/Projects/education/workshops/modular-workshops/langgraph.json is valid. (1 graph found)


## 5. Deploy to LangSmith (Optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes — feel free to start Module 4 while it deploys.

> Requires a `LANGSMITH_API_KEY` with deployment permissions (a `lsv2_sk_...` service key, not a personal token). On Apple Silicon, local builds need Docker Buildx — if you don't have Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build (skips local Docker entirely)
- `--no-wait` — don't block waiting for status


In [5]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name modular-workshops-deep-agent --no-input


Note: 'langgraph deploy' is in beta. Expect frequent updates and improvements.

⚠️  Security Recommendation: Consider switching to Wolfi Linux for enhanced security.
   Wolfi is a security-oriented, minimal Linux distribution designed for containers.
   To switch, add '"image_distro": "wolfi"' to your langgraph.json config file.
Skipping reserved env var: LANGSMITH_ENDPOINT
Skipping reserved env var: LANGSMITH_API_KEY
Skipping reserved env var: LANGSMITH_PROJECT
Docker is installed but not running.
Start Docker and try again.
Using remote build instead.

1. Looking up deployment 'modular-workshops-deep-agent'
   Found existing deployment (ID: cdbc08df-7360-4666-8fa0-7fb03867a11d)
2. Creating source archive
   Archive created (2.1 MB)
3. Requesting upload URL
4. Uploading source
   Uploading (2.1 MB)... 100%
5. Triggering remote build
   View status: https://smith.langchain.com/o/58636190-0252-4526-9dc7-0b09b37b499c/host/deployments/cdbc08df-7360-4666-8fa0-7fb03867a11d
   QUEUED... (1m 

## 6. What You Get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


## Deploy Recap

| File | Purpose |
|------|---------|
| `langgraph.json` | LangGraph deploy config (dependencies, graphs, env file) |
| `agent.py` | The deployable `agent` object (built with `create_deep_agent`) |
| `AGENTS.md` | Agent identity and instructions (loaded via `memory=`) |
| `skills/` | On-demand capability modules (loaded via `skills=`) |
| `pyproject.toml` | Python dependencies for the deploy image |

**Useful follow-ups:**
- `langgraph deploy list` — list your deployments
- `langgraph deploy logs <id>` — tail logs
- `langgraph deploy revisions list <id>` — see revision history
- Re-run `langgraph deploy` to push a new revision (the CLI finds the existing deployment by name)

**Next:** Module 4 covers how to trace, evaluate, and monitor the deployed agent with LangSmith.
